## Architecture Overview

The Travel Reimbursement Approval Agent follows a structured AI workflow to evaluate employee reimbursement claims using Retrieval-Augmented Generation (RAG), business rule validation, and LLM reasoning.

### Workflow

1. Claim Intake
2. Document Processing
3. Vector Database (Knowledge Base)
4. Policy Retrieval (RAG)
5. Receipt Validation Tool
6. Expense Validation Tool
7. Approval Threshold Tool
8. LLM Decision
9. Output Validation
10. Dashboard & Final JSON Output

### Final Decisions

- APPROVE
- PARTIAL_APPROVE
- REJECT
- MANUAL_REVIEW

# AI Developer Candidate Assignment

# Travel Reimbursement Approval Agent

## Objective
Develop an AI-powered Travel Reimbursement Approval Agent that evaluates employee reimbursement claims using company policies, business rules, Retrieval-Augmented Generation (RAG), and LLM reasoning.

## Technology Stack
- Python
- OpenAI GPT
- LangChain
- LangGraph
- Chroma
- Pydantic
- Pandas
- Matplotlib

## Workflow
Claim Intake → RAG → Business Rule Validation → LLM Decision → Output Validation → Dashboard

## Expected Output
For each claim, generate:
- Decision
- Approved & Deducted Amount
- Missing Documents
- Policy References
- Confidence Score
- Explanation

In [2]:
# ==========================================================
# Block 3 : Environment Setup
# ==========================================================

import sys
import subprocess
import importlib.util

print("=" * 60)
print("Travel Reimbursement Approval Agent - Environment Setup")
print("=" * 60)

# Minimum Python Version
MIN_PYTHON = (3, 10)

if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        f"Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]} or higher is required."
    )

print(f"✓ Python Version : {sys.version.split()[0]}")

# Required Packages
required_packages = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_openai": "langchain-openai",
    "langchain_chroma": "langchain-chroma",
    "chromadb": "chromadb",
    "openai": "openai",
    "pydantic": "pydantic",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "pypdf": "pypdf",
    "python_docx": "python-docx"
}

print("\nChecking required packages...\n")

for module_name, package_name in required_packages.items():

    if importlib.util.find_spec(module_name) is None:

        print(f"Installing {package_name}...")

        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", package_name],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
        )

        if result.returncode == 0:
            print(f"✓ {package_name} installed successfully")
        else:
            print(f"✗ Failed to install {package_name}")
            print(result.stderr)

    else:
        print(f"✓ {package_name} already installed")

print("\nEnvironment setup completed successfully.")

Travel Reimbursement Approval Agent - Environment Setup
✓ Python Version : 3.14.4

Checking required packages...

✓ langchain already installed
✓ langgraph already installed
✓ langchain-openai already installed
✓ langchain-chroma already installed
✓ chromadb already installed
✓ openai already installed
✓ pydantic already installed
✓ pandas already installed
✓ matplotlib already installed
✓ pypdf already installed
Installing python-docx...
✓ python-docx installed successfully

Environment setup completed successfully.


In [3]:
# ==========================================================
# Block 4 : Import Libraries
# ==========================================================

# ----------------------------------------------------------
# Standard Library
# ----------------------------------------------------------
import os
import json
import warnings
from typing import TypedDict, List, Dict, Any

# ----------------------------------------------------------
# File Handling
# ----------------------------------------------------------
from pathlib import Path
from pypdf import PdfReader
from docx import Document as DocxDocument

# ----------------------------------------------------------
# Data Processing & Visualization
# ----------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------
# Data Validation
# ----------------------------------------------------------
from pydantic import BaseModel, Field

# ----------------------------------------------------------
# LangChain
# ----------------------------------------------------------
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ----------------------------------------------------------
# OpenAI
# ----------------------------------------------------------
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ----------------------------------------------------------
# Vector Database
# ----------------------------------------------------------
from langchain_chroma import Chroma

# ----------------------------------------------------------
# LangGraph
# ----------------------------------------------------------
from langgraph.graph import StateGraph, END

# ----------------------------------------------------------
# Notebook Configuration
# ----------------------------------------------------------
warnings.filterwarnings("ignore")

print("=" * 65)
print("✓ All required libraries imported successfully.")
print("=" * 65)

✓ All required libraries imported successfully.


In [ ]:
# ==========================================================
# Block 5 : Configuration
# ==========================================================

import getpass

print("=" * 65)
print("Travel Reimbursement Approval Agent - Configuration")
print("=" * 65)

# ----------------------------------------------------------
# OpenAI Credentials
# ----------------------------------------------------------

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API Key: ")

OPENAI_BASE_URL = input(
     "Enter OpenAI Base URL (Press Enter for default): "
 ).strip()

if not OPENAI_BASE_URL:
    OPENAI_BASE_URL = "https://api.openai.com/v1"

# ----------------------------------------------------------
# Model Configuration
# ----------------------------------------------------------

LLM_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ----------------------------------------------------------
# RAG Configuration
# ----------------------------------------------------------

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K_RESULTS = 3

print("\n✓ Configuration Loaded Successfully\n")

print(f"Base URL         : {OPENAI_BASE_URL}")
print(f"LLM Model        : {LLM_MODEL}")
print(f"Embedding Model  : {EMBEDDING_MODEL}")
print(f"Chunk Size       : {CHUNK_SIZE}")
print(f"Chunk Overlap    : {CHUNK_OVERLAP}")
print(f"Top K Retrieval  : {TOP_K_RESULTS}")

Travel Reimbursement Approval Agent - Configuration

✓ Configuration Loaded Successfully

Base URL         : https://openai.vocareum.com/v1
LLM Model        : gpt-4.1-mini
Embedding Model  : text-embedding-3-small
Chunk Size       : 500
Chunk Overlap    : 100
Top K Retrieval  : 3


In [5]:
# ==========================================================
# Block 6 : Environment Validation
# ==========================================================

print("=" * 65)
print("Travel Reimbursement Approval Agent - Environment Validation")
print("=" * 65)

try:

    # Test LLM Connection
    llm = ChatOpenAI(
        model=LLM_MODEL,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
        temperature=0
    )

    response = llm.invoke("Reply with only the word: SUCCESS")

    print("✓ OpenAI Connection : Successful")
    print(f"✓ Response          : {response.content}")

except Exception as ex:

    print("❌ OpenAI Connection Failed")
    print(f"Reason : {ex}")

    raise SystemExit("Please verify your API Key or Base URL.")

print("\nEnvironment validation completed successfully.")

Travel Reimbursement Approval Agent - Environment Validation
✓ OpenAI Connection : Successful
✓ Response          : SUCCESS

Environment validation completed successfully.


## Input Data

The Travel Reimbursement Approval Agent uses two primary inputs:

### 1. Travel Reimbursement Policy
The reimbursement policy defines the business rules that govern claim processing, including:

- Eligible expense categories
- Reimbursement limits
- Receipt requirements
- Approval thresholds
- Policy references

This policy will be converted into a knowledge base and indexed using a Vector Database for Retrieval-Augmented Generation (RAG).

---

### 2. Employee Travel Claims

Five sample reimbursement claims are used to demonstrate different approval scenarios, including:

- Fully Approved
- Partially Approved
- Rejected
- Manual Review
- Missing Documentation

These sample claims are stored directly within the notebook to ensure the solution is self-contained and easy to execute.

In [6]:
# ==========================================================
# Block 8 : Load Policy Documents
# ==========================================================

from pathlib import Path

print("=" * 65)
print("Loading Policy Documents")
print("=" * 65)

# ----------------------------------------------------------
# Documents Folder
# ----------------------------------------------------------
DOCUMENTS_FOLDER = Path("documents")

if not DOCUMENTS_FOLDER.exists():
    raise FileNotFoundError(
        f"Documents folder not found: {DOCUMENTS_FOLDER.resolve()}"
    )

# ----------------------------------------------------------
# Supported File Types
# ----------------------------------------------------------
SUPPORTED_FILES = (
    "*.pdf",
    "*.docx",
    "*.txt"
)

policy_files = []

for pattern in SUPPORTED_FILES:
    policy_files.extend(DOCUMENTS_FOLDER.glob(pattern))

if not policy_files:
    raise FileNotFoundError(
        "No supported documents found inside the 'documents' folder."
    )

print(f"✓ Documents Found : {len(policy_files)}\n")

for i, file in enumerate(policy_files, start=1):
    print(f"{i}. {file.name}")

print("\nPolicy document discovery completed successfully.")

Loading Policy Documents
✓ Documents Found : 1

1. Appendix A — Travel Reimbursement Policy.docx

Policy document discovery completed successfully.


In [7]:
# ==========================================================
# Block 9 : Read Policy Documents
# ==========================================================

print("=" * 65)
print("Reading Policy Documents")
print("=" * 65)

langchain_documents = []

for file in policy_files:

    try:

        file_extension = file.suffix.lower()

        # --------------------------------------------------
        # Read PDF
        # --------------------------------------------------
        if file_extension == ".pdf":

            reader = PdfReader(file)

            text = ""

            for page in reader.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        # --------------------------------------------------
        # Read DOCX
        # --------------------------------------------------
        elif file_extension == ".docx":

            doc = DocxDocument(file)

            text = "\n".join(
                paragraph.text
                for paragraph in doc.paragraphs
            )

        # --------------------------------------------------
        # Read TXT
        # --------------------------------------------------
        elif file_extension == ".txt":

            text = file.read_text(
                encoding="utf-8"
            )

        else:
            continue

        # --------------------------------------------------
        # Create LangChain Document
        # --------------------------------------------------
        langchain_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": file.name,
                    "file_type": file_extension
                }
            )
        )

        print(f"✓ Loaded : {file.name}")

    except Exception as ex:

        print(f"✗ Failed : {file.name}")
        print(f"Reason   : {ex}")

print("\n" + "=" * 65)
print(f"Total Documents Loaded : {len(langchain_documents)}")
print("=" * 65)

Reading Policy Documents
✓ Loaded : Appendix A — Travel Reimbursement Policy.docx

Total Documents Loaded : 1


In [8]:
# ==========================================================
# Block 10 : Text Chunking
# ==========================================================

print("=" * 65)
print("Splitting Documents into Chunks")
print("=" * 65)

# ----------------------------------------------------------
# Initialize Text Splitter
# ----------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

# ----------------------------------------------------------
# Split Documents
# ----------------------------------------------------------
document_chunks = text_splitter.split_documents(
    langchain_documents
)

print(f"✓ Documents           : {len(langchain_documents)}")
print(f"✓ Total Chunks        : {len(document_chunks)}")
print(f"✓ Chunk Size          : {CHUNK_SIZE}")
print(f"✓ Chunk Overlap       : {CHUNK_OVERLAP}")

# ----------------------------------------------------------
# Preview First Chunk
# ----------------------------------------------------------
if document_chunks:
    print("\nFirst Chunk Preview:\n")
    print(document_chunks[0].page_content[:500])

Splitting Documents into Chunks
✓ Documents           : 1
✓ Total Chunks        : 9
✓ Chunk Size          : 500
✓ Chunk Overlap       : 100

First Chunk Preview:

Appendix A — Travel Reimbursement Policy 
Mock policy for this assignment. No real company or employee data. Every rule has a stable id (POL-*) you should cite in your output where relevant. 
1. Eligible & Ineligible Categories 
POL-CAT-01 — Eligible categories — Reimbursable when incurred for a documented business purpose: 
Airfare (economy class only — see POL-AIR-01) 
Lodging (hotel room charges) 
Meals (subject to per-diem limits — see POL-PD-01)


In [9]:
# ==========================================================
# Block 11 : Create Embeddings
# ==========================================================

print("=" * 65)
print("Creating OpenAI Embeddings")
print("=" * 65)

# ----------------------------------------------------------
# Initialize Embedding Model
# ----------------------------------------------------------
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)

print(f"✓ Embedding Model : {EMBEDDING_MODEL}")
print("✓ Embedding model initialized successfully.")

Creating OpenAI Embeddings
✓ Embedding Model : text-embedding-3-small
✓ Embedding model initialized successfully.


In [10]:
# ==========================================================
# Block 12 : Build Vector Database
# ==========================================================

print("=" * 65)
print("Building Chroma Vector Database")
print("=" * 65)

# ----------------------------------------------------------
# Create Vector Database
# ----------------------------------------------------------
vector_db = Chroma.from_documents(
    documents=document_chunks,
    embedding=embeddings
)

print(f"✓ Total Chunks Indexed : {len(document_chunks)}")
print("✓ Chroma Vector Database created successfully.")

Building Chroma Vector Database
✓ Total Chunks Indexed : 9
✓ Chroma Vector Database created successfully.


In [11]:
# ==========================================================
# Block 13 : Create Retriever
# ==========================================================

print("=" * 65)
print("Creating Document Retriever")
print("=" * 65)

# ----------------------------------------------------------
# Create Retriever
# ----------------------------------------------------------
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": TOP_K_RESULTS
    }
)

print("✓ Retriever created successfully.")
print(f"✓ Search Type        : Similarity Search")
print(f"✓ Top K Results      : {TOP_K_RESULTS}")

Creating Document Retriever
✓ Retriever created successfully.
✓ Search Type        : Similarity Search
✓ Top K Results      : 3


## Business Tools

After retrieving relevant policy information using RAG, the AI agent uses a set of business tools to validate each reimbursement claim before making a final decision.

### Tools

1. **Policy Lookup Tool**
   - Retrieves relevant policy sections from the Vector Database.

2. **Receipt Validation Tool**
   - Verifies whether mandatory receipts are available.

3. **Expense Validation Tool**
   - Validates claimed amounts against reimbursement limits.

4. **Approval Threshold Tool**
   - Determines whether manager approval is required based on the claim amount.

5. **Output Validation Tool**
   - Ensures the final response follows the required output schema.

In [ ]:
# ==========================================================
# Block 15 : Policy Lookup Tool
# ==========================================================

print("=" * 65)
print("Creating Policy Lookup Tool")
print("=" * 65)

def policy_lookup_tool(query: str, top_k: int = TOP_K_RESULTS) -> list:
    """
    Retrieve relevant policy documents from the Vector Database.
    """

    results = retriever.invoke(query)

    return [
        {
            "content": doc.page_content,
            "source": doc.metadata.get("source", "Unknown")
        }
        for doc in results
    ]

print("✓ Policy Lookup Tool created successfully.")

Creating Policy Lookup Tool
✓ Policy Lookup Tool created successfully.


In [13]:
# ==========================================================
# Block 16 : Receipt Validation Tool
# ==========================================================

print("=" * 65)
print("Creating Receipt Validation Tool")
print("=" * 65)

def receipt_validation_tool(claim: dict) -> dict:
    """
    Validate receipt availability for a claim.
    """

    receipts = claim.get("receipts", [])

    if receipts:
        return {
            "status": True,
            "message": "All required receipts are available."
        }

    return {
        "status": False,
        "message": "Required receipts are missing."
    }

print("✓ Receipt Validation Tool created successfully.")

Creating Receipt Validation Tool
✓ Receipt Validation Tool created successfully.


In [14]:
# ==========================================================
# Block 17 : Expense Validation Tool
# ==========================================================

print("=" * 65)
print("Creating Expense Validation Tool")
print("=" * 65)

def expense_validation_tool(
    claimed_amount: float,
    policy_limit: float
) -> dict:
    """
    Validate the claimed amount against the policy limit.
    """

    approved_amount = min(claimed_amount, policy_limit)
    deducted_amount = max(0, claimed_amount - policy_limit)

    return {
        "status": claimed_amount <= policy_limit,
        "claimed_amount": claimed_amount,
        "policy_limit": policy_limit,
        "approved_amount": approved_amount,
        "deducted_amount": deducted_amount
    }

print("✓ Expense Validation Tool created successfully.")

Creating Expense Validation Tool
✓ Expense Validation Tool created successfully.


In [15]:
# ==========================================================
# Block 18 : Approval Threshold Tool
# ==========================================================

print("=" * 65)
print("Creating Approval Threshold Tool")
print("=" * 65)

def approval_threshold_tool(
    claimed_amount: float,
    approval_threshold: float
) -> dict:
    """
    Check whether manager approval is required.
    """

    if claimed_amount > approval_threshold:
        return {
            "requires_approval": True,
            "message": "Manager approval required."
        }

    return {
        "requires_approval": False,
        "message": "Manager approval not required."
    }

print("✓ Approval Threshold Tool created successfully.")

Creating Approval Threshold Tool
✓ Approval Threshold Tool created successfully.


In [16]:
# ==========================================================
# Block 19 : Output Validation Tool
# ==========================================================

print("=" * 65)
print("Creating Output Validation Tool")
print("=" * 65)

def output_validation_tool(result: dict) -> dict:
    """
    Validate the final response before returning it.
    """

    required_fields = [
        "claim_id",
        "decision",
        "approved_amount",
        "deducted_amount",
        "missing_documents",
        "policy_references",
        "confidence_score",
        "explanation",
        "tools_used"
    ]

    missing_fields = [
        field for field in required_fields
        if field not in result
    ]

    return {
        "is_valid": len(missing_fields) == 0,
        "missing_fields": missing_fields
    }

print("✓ Output Validation Tool created successfully.")

Creating Output Validation Tool
✓ Output Validation Tool created successfully.


## AI Agent

The AI Agent combines the retrieved policy information and business tool results to make the final reimbursement decision.

### Responsibilities

- Retrieve relevant policy sections
- Validate receipts
- Validate expense limits
- Check approval thresholds
- Generate the final reimbursement decision
- Return a structured JSON response

In [17]:
# ==========================================================
# Block 21 : Prompt Template
# ==========================================================

print("=" * 65)
print("Creating Prompt Template")
print("=" * 65)

prompt = ChatPromptTemplate.from_template("""
You are an AI Travel Reimbursement Approval Agent.

Your responsibilities are to:

1. Review the employee reimbursement claim.
2. Analyze the retrieved travel policy.
3. Validate:
   - Receipt availability
   - Expense limits
   - Approval threshold
4. Make a final decision.

Possible Decisions:
- APPROVE
- PARTIAL_APPROVE
- REJECT
- MANUAL_REVIEW

Return ONLY a valid JSON in the following format:

{{
    "claim_id": "",
    "decision": "",
    "reason": "",
    "request_amount":0,
    "approved_amount": 0,
    "deducted_amount": 0,
    "missing_documents": [],
    "policy_references": [],
    "confidence_score": 0.0,
    "explanation": "",
    "tools_used": []
}}

Claim:
{claim}

Retrieved Policy:
{policy}

Receipt Validation:
{receipt_validation}

Expense Validation:
{expense_validation}

Approval Validation:
{approval_validation}
""")

print("✓ Prompt Template created successfully.")

Creating Prompt Template
✓ Prompt Template created successfully.


In [18]:
# ==========================================================
# Block 22 : Initialize LLM
# ==========================================================

print("=" * 65)
print("Initializing OpenAI LLM")
print("=" * 65)

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    temperature=0
)

print(f"✓ Model        : {LLM_MODEL}")
print("✓ LLM initialized successfully.")

Initializing OpenAI LLM
✓ Model        : gpt-4.1-mini
✓ LLM initialized successfully.


In [19]:
# ==========================================================
# Block 23 : Create LLM Chain
# ==========================================================

print("=" * 65)
print("Creating LLM Chain")
print("=" * 65)

llm_chain = prompt | llm

print("✓ LLM Chain created successfully.")

Creating LLM Chain
✓ LLM Chain created successfully.


In [20]:
# ==========================================================
# Block 24 : Define Agent State
# ==========================================================

print("=" * 65)
print("Defining Agent State")
print("=" * 65)

class AgentState(TypedDict):
    claim: Dict[str, Any]
    retrieved_policy: List[Dict[str, Any]]
    receipt_validation: Dict[str, Any]
    expense_validation: Dict[str, Any]
    approval_validation: Dict[str, Any]
    llm_response: Dict[str, Any]
    final_result: Dict[str, Any]

print("✓ Agent State defined successfully.")

Defining Agent State
✓ Agent State defined successfully.


In [21]:
# ==========================================================
# Block 25 : Create LangGraph Workflow
# ==========================================================

print("=" * 65)
print("Creating LangGraph Workflow")
print("=" * 65)

# ----------------------------------------------------------
# Create State Graph
# ----------------------------------------------------------
workflow = StateGraph(AgentState)

print("✓ LangGraph workflow created successfully.")

Creating LangGraph Workflow
✓ LangGraph workflow created successfully.


In [22]:
# ==========================================================
# Block 26 : Define Workflow Nodes
# ==========================================================

print("=" * 65)
print("Defining Workflow Nodes")
print("=" * 65)

# ----------------------------------------------------------
# Policy Lookup Node
# ----------------------------------------------------------
def policy_lookup_node(state: AgentState):

    state["retrieved_policy"] = policy_lookup_tool(
        state["claim"]["employee_query"]
    )

    return state


# ----------------------------------------------------------
# Receipt Validation Node
# ----------------------------------------------------------
def receipt_validation_node(state: AgentState):

    state["receipt_validation"] = receipt_validation_tool(
        state["claim"]
    )

    return state


# ----------------------------------------------------------
# Expense Validation Node
# ----------------------------------------------------------
def expense_validation_node(state: AgentState):

    state["expense_validation"] = expense_validation_tool(
        claimed_amount=state["claim"]["claimed_amount"],
        policy_limit=state["claim"]["policy_limit"]
    )

    return state


# ----------------------------------------------------------
# Approval Validation Node
# ----------------------------------------------------------
def approval_validation_node(state: AgentState):

    state["approval_validation"] = approval_threshold_tool(
        claimed_amount=state["claim"]["claimed_amount"],
        approval_threshold=state["claim"]["approval_threshold"]
    )

    return state

print("✓ Workflow nodes defined successfully.")

Defining Workflow Nodes
✓ Workflow nodes defined successfully.


In [23]:
# ==========================================================
# Block 27 : Register Workflow Nodes
# ==========================================================

print("=" * 65)
print("Registering Workflow Nodes")
print("=" * 65)

# ----------------------------------------------------------
# Register Nodes
# ----------------------------------------------------------
workflow.add_node("policy_lookup", policy_lookup_node)

workflow.add_node("receipt_validation", receipt_validation_node)

workflow.add_node("expense_validation", expense_validation_node)

workflow.add_node("approval_threshold", approval_validation_node)

print("✓ Policy Lookup Node Registered")
print("✓ Receipt Validation Node Registered")
print("✓ Expense Validation Node Registered")
print("✓ Approval Threshold Node Registered")

print("\nWorkflow nodes registered successfully.")

Registering Workflow Nodes
✓ Policy Lookup Node Registered
✓ Receipt Validation Node Registered
✓ Expense Validation Node Registered
✓ Approval Threshold Node Registered

Workflow nodes registered successfully.


In [24]:
# ==========================================================
# Block 28 : Add Workflow Edges
# ==========================================================

print("=" * 65)
print("Adding Workflow Edges")
print("=" * 65)

# ----------------------------------------------------------
# Set Entry Point
# ----------------------------------------------------------
workflow.set_entry_point("policy_lookup")

# ----------------------------------------------------------
# Define Workflow Flow
# ----------------------------------------------------------
workflow.add_edge("policy_lookup", "receipt_validation")

workflow.add_edge("receipt_validation", "expense_validation")

workflow.add_edge("expense_validation", "approval_threshold")

workflow.add_edge("approval_threshold", END)

print("✓ Entry Point        : policy_lookup")
print("✓ Exit Point         : END")
print("✓ Workflow edges added successfully.")

Adding Workflow Edges
✓ Entry Point        : policy_lookup
✓ Exit Point         : END
✓ Workflow edges added successfully.


In [25]:
# ==========================================================
# Block 29 : LLM Decision Node
# ==========================================================

print("=" * 65)
print("Defining LLM Decision Node")
print("=" * 65)

def llm_decision_node(state: AgentState):

    response = llm_chain.invoke({
        "claim": state["claim"],
        "policy": state["retrieved_policy"],
        "receipt_validation": state["receipt_validation"],
        "expense_validation": state["expense_validation"],
        "approval_validation": state["approval_validation"]
    })

    state["llm_response"] = json.loads(response.content)

    return state

print("✓ LLM Decision Node defined successfully.")

Defining LLM Decision Node
✓ LLM Decision Node defined successfully.


In [26]:
# ==========================================================
# Block 30 : Register LLM Node
# ==========================================================

print("=" * 65)
print("Registering LLM Decision Node")
print("=" * 65)

# ----------------------------------------------------------
# Register LLM Node
# ----------------------------------------------------------
workflow.add_node("llm_decision", llm_decision_node)

# ----------------------------------------------------------
# Update Workflow
# ----------------------------------------------------------
workflow.add_edge("approval_threshold", "llm_decision")

print("✓ LLM Decision Node Registered")
print("✓ Workflow updated successfully.")

Registering LLM Decision Node
✓ LLM Decision Node Registered
✓ Workflow updated successfully.


In [27]:
# ==========================================================
# Block 31 : Output Validation Node
# ==========================================================

print("=" * 65)
print("Defining Output Validation Node")
print("=" * 65)

def output_validation_node(state: AgentState):

    state["final_result"] = output_validation_tool(
        state["llm_response"]
    )

    return state

print("✓ Output Validation Node defined successfully.")

Defining Output Validation Node
✓ Output Validation Node defined successfully.


In [28]:
# ==========================================================
# Block 32 : Register Output Validation Node
# ==========================================================

print("=" * 65)
print("Registering Output Validation Node")
print("=" * 65)

# ----------------------------------------------------------
# Register Output Validation Node
# ----------------------------------------------------------
workflow.add_node("output_validation", output_validation_node)

# ----------------------------------------------------------
# Complete Workflow
# ----------------------------------------------------------
workflow.add_edge("llm_decision", "output_validation")

workflow.add_edge("output_validation", END)

print("✓ Output Validation Node Registered")
print("✓ Workflow completed successfully.")

Registering Output Validation Node
✓ Output Validation Node Registered
✓ Workflow completed successfully.


In [29]:
# ==========================================================
# Block 33 : Compile LangGraph Workflow
# ==========================================================

print("=" * 65)
print("Compiling LangGraph Workflow")
print("=" * 65)

# ----------------------------------------------------------
# Compile Workflow
# ----------------------------------------------------------
app = workflow.compile()

print("✓ LangGraph workflow compiled successfully.")

Compiling LangGraph Workflow
✓ LangGraph workflow compiled successfully.


In [30]:
# ==========================================================
# Block 34 : Sample Travel Claims
# ==========================================================

print("=" * 65)
print("Loading Sample Travel Claims")
print("=" * 65)

sample_claims = [

    # ------------------------------------------------------
    # CLM001 - Valid Claim
    # ------------------------------------------------------
    {
        "claim_id": "CLM001",
        "employee_query": "Business trip to Bangalore.",
        "claimed_amount": 8500,
        "policy_limit": 10000,
        "approval_threshold": 15000,
        "receipts": ["flight.pdf", "hotel.pdf"]
    },

    # ------------------------------------------------------
    # CLM002 - Exceeds Policy Limit
    # ------------------------------------------------------
    {
        "claim_id": "CLM002",
        "employee_query": "Hotel reimbursement exceeded policy limit.",
        "claimed_amount": 18000,
        "policy_limit": 12000,
        "approval_threshold": 25000,
        "receipts": ["hotel.pdf"]
    },

    # ------------------------------------------------------
    # CLM003 - Missing Receipts
    # ------------------------------------------------------
    {
        "claim_id": "CLM003",
        "employee_query": "Meal reimbursement without receipt.",
        "claimed_amount": 1200,
        "policy_limit": 1500,
        "approval_threshold": 15000,
        "receipts": []
    },

    # ------------------------------------------------------
    # CLM004 - Requires Manager Approval
    # ------------------------------------------------------
    {
        "claim_id": "CLM004",
        "employee_query": "International business travel.",
        "claimed_amount": 32000,
        "policy_limit": 40000,
        "approval_threshold": 15000,
        "receipts": [
            "flight.pdf",
            "hotel.pdf",
            "visa.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM005 - Exactly at Policy Limit
    # ------------------------------------------------------
    {
        "claim_id": "CLM005",
        "employee_query": "Domestic travel reimbursement.",
        "claimed_amount": 10000,
        "policy_limit": 10000,
        "approval_threshold": 15000,
        "receipts": [
            "flight.pdf",
            "hotel.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM006 - Exactly at Approval Threshold
    # ------------------------------------------------------
    {
        "claim_id": "CLM006",
        "employee_query": "Client meeting travel expenses.",
        "claimed_amount": 15000,
        "policy_limit": 18000,
        "approval_threshold": 15000,
        "receipts": [
            "flight.pdf",
            "hotel.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM007 - Zero Amount
    # ------------------------------------------------------
    {
        "claim_id": "CLM007",
        "employee_query": "Travel claim with zero amount.",
        "claimed_amount": 0,
        "policy_limit": 5000,
        "approval_threshold": 15000,
        "receipts": [
            "receipt.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM008 - Unsupported Expense
    # ------------------------------------------------------
    {
        "claim_id": "CLM008",
        "employee_query": "Luxury spa reimbursement.",
        "claimed_amount": 4500,
        "policy_limit": 0,
        "approval_threshold": 15000,
        "receipts": [
            "spa_invoice.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM009 - Multiple Expenses
    # ------------------------------------------------------
    {
        "claim_id": "CLM009",
        "employee_query": "Flight, hotel and meal reimbursement.",
        "claimed_amount": 13500,
        "policy_limit": 15000,
        "approval_threshold": 20000,
        "receipts": [
            "flight.pdf",
            "hotel.pdf",
            "meals.pdf"
        ]
    },

    # ------------------------------------------------------
    # CLM010 - Missing Mandatory Information
    # ------------------------------------------------------
    {
        "claim_id": "CLM010",
        "employee_query": "",
        "claimed_amount": 9000,
        "policy_limit": 10000,
        "approval_threshold": 15000,
        "receipts": []
    }

]

print(f"✓ Total Claims Loaded : {len(sample_claims)}")

Loading Sample Travel Claims
✓ Total Claims Loaded : 10


In [31]:
# ==========================================================
# Block 35 : Execute LangGraph Workflow
# ==========================================================

print("=" * 65)
print("Executing LangGraph Workflow")
print("=" * 65)

results = []

for claim in sample_claims:

    state = {
        "claim": claim,
        "retrieved_policy": [],
        "receipt_validation": {},
        "expense_validation": {},
        "approval_validation": {},
        "llm_response": {},
        "final_result": {}
    }

    result = app.invoke(state)

    results.append(result)

print(f"✓ Claims Processed : {len(results)}")

Executing LangGraph Workflow
✓ Claims Processed : 10


In [32]:
# ==========================================================
# Block 36 : Display Results
# ==========================================================

print("=" * 65)
print("Travel Reimbursement Results")
print("=" * 65)

for result in results:

    print(json.dumps(result["llm_response"], indent=4))

    print("-" * 65)

Travel Reimbursement Results
{
    "claim_id": "CLM001",
    "decision": "APPROVE",
    "reason": "All required receipts are provided, claimed amount is within policy limits, and approval threshold is not exceeded.",
    "request_amount": 8500,
    "approved_amount": 8500,
    "deducted_amount": 0,
    "missing_documents": [],
    "policy_references": [
        "POL-CAT-01",
        "POL-PD-01",
        "POL-PD-02",
        "POL-PD-03"
    ],
    "confidence_score": 1.0,
    "explanation": "The claim meets all policy requirements: receipts for flight and hotel are submitted, the claimed amount of $8500 is below the $10000 policy limit, and no additional managerial approval is needed as the amount is below the $15000 threshold.",
    "tools_used": [
        "Receipt Validation",
        "Expense Validation",
        "Approval Validation"
    ]
}
-----------------------------------------------------------------
{
    "claim_id": "CLM002",
    "decision": "PARTIAL_APPROVE",
    "reason": 

In [33]:
# ==========================================================
# Block 37 : Dashboard
# ==========================================================

print("=" * 65)
print("Travel Reimbursement Dashboard")
print("=" * 65)

dashboard = pd.DataFrame([
    {
        "Claim ID": r["llm_response"]["claim_id"],
        "Decision": r["llm_response"]["decision"],
        "Request Amount":r["llm_response"]["request_amount"],
        "Approved Amount": r["llm_response"]["approved_amount"],
        "Deducted Amount": r["llm_response"]["deducted_amount"],
        "Confidence": r["llm_response"]["confidence_score"],
        "Reason": r["llm_response"]["reason"],
    }
    for r in results
])

display(
    dashboard.style
    .hide(axis="index")
    .format({
        "Approved Amount": "₹{:,.0f}",
        "Deducted Amount": "₹{:,.0f}",
        "Confidence": "{:.0%}"
    })
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#4CAF50"),
                   ("color", "white"),
                   ("text-align", "center")]},
        {"selector": "td",
         "props": [("padding", "8px"),
                   ("vertical-align", "middle")]}
    ])
    .set_properties(subset=["Claim ID", "Decision", "Reason"],
                    **{"text-align": "left"})
    .set_properties(subset=["Approved Amount", "Deducted Amount", "Confidence"],
                    **{"text-align": "center"})
)

print(f"\nTotal Claims : {len(dashboard)}")

Travel Reimbursement Dashboard


Claim ID,Decision,Request Amount,Approved Amount,Deducted Amount,Confidence,Reason
CLM001,APPROVE,8500,"₹8,500",₹0,100%,"All required receipts are provided, claimed amount is within policy limits, and approval threshold is not exceeded."
CLM002,PARTIAL_APPROVE,18000,"₹12,000","₹6,000",100%,"Claimed hotel expense exceeds the policy limit; amount above $12,000 is deducted as per policy POL-PD-02."
CLM003,MANUAL_REVIEW,1200,₹0,₹0,95%,Missing required receipt for meal expense above $25 as per POL-RCT-01 and POL-RCT-02.
CLM004,MANUAL_REVIEW,32000,₹0,₹0,95%,Claim amount exceeds approval threshold requiring managerial approval despite meeting policy limits and having all receipts.
CLM005,APPROVE,10000,"₹10,000",₹0,100%,"All receipts are provided, claimed amount is within policy limits, and approval threshold is not exceeded."
CLM006,APPROVE,15000,"₹15,000",₹0,100%,Claimed amount is within policy limits and approval threshold; all required receipts are provided.
CLM007,REJECT,0,₹0,₹0,100%,"Claimed amount is zero, no reimbursement applicable."
CLM008,REJECT,4500,₹0,"₹4,500",100%,Expense is for a luxury spa which is explicitly listed as a never reimbursable item in the travel policy.
CLM009,APPROVE,13500,"₹13,500",₹0,100%,"All receipts are provided, claimed amount is within policy limits, and approval threshold is not exceeded."
CLM010,MANUAL_REVIEW,9000,₹0,₹0,90%,"Missing required receipts for the claimed expenses, which is against policy POL-RCT-02 requiring receipts for reimbursement."



Total Claims : 10


# Conclusion

The Travel Reimbursement Approval Agent demonstrates an end-to-end AI workflow for processing employee reimbursement claims using Retrieval-Augmented Generation (RAG), business rule validation, LangGraph orchestration, and OpenAI Large Language Models.

### Key Features

- Document-based policy retrieval using RAG
- Multi-format document support (PDF, DOCX, TXT)
- Automatic business rule validation
- AI-assisted reimbursement decisions
- Structured JSON output
- LangGraph workflow orchestration
- Interactive dashboard for result visualization

---

# Future Enhancements

- Persistent Vector Database
- Multi-agent architecture (CrewAI / LangGraph Supervisor)
- Human-in-the-loop approval workflow
- OCR support for scanned receipts
- Web-based user interface
- Database integration
- Email/Notification integration
- Authentication and Role-Based Access Control (RBAC)